In [1]:
import pandas as pd
import xgboost as xgb
from sklearn.feature_selection import SelectFromModel
import pickle
import warnings
import numpy as np

warnings.filterwarnings('ignore')

# Load and prepare datasets
df_train = pd.read_csv('Datasets_52/train_K_6months_top_15.csv').drop('PERSON_ID', axis=1)

# Class counts
class_counts = {
    "NCHS_15": 4155,
    "other": 2661,
    "NCHS_23": 2192,
    "NCHS_52": 1044,
    "NCHS_45": 1042,
    "NCHS_25": 612,
    "NCHS_36": 364,
    "NCHS_30": 353,
    "NCHS_18": 306,
    "NCHS_38": 194,
    "NCHS_28": 188,
    "NCHS_7": 157,
    "NCHS_46": 153,
    "NCHS_21": 131,
    "NCHS_24": 129,
    "NCHS_22": 115
}

class_list = list(class_counts.keys())

X_train = df_train.iloc[:, :-len(class_list)]
y_train = df_train.iloc[:, -len(class_list):]
y_train_idx = y_train.values.argmax(axis=1)

# Train XGBoost model
xgb_model = xgb.XGBClassifier(objective='multi:softprob', num_class=len(class_list), seed=42)
xgb_model.fit(X_train, y_train_idx)

# Get feature importances
importances = xgb_model.feature_importances_
indices = np.argsort(importances)[-200:]  # Get indices of top 200 features

# Select top 200 features
selected_features = X_train.columns[indices]
X_train_selected = X_train[selected_features]

# Save selected features
with open('selected_features.pkl', 'wb') as f:
    pickle.dump(selected_features.tolist(), f)

# Save removed features to CSV
removed_features = X_train.columns.difference(selected_features)
removed_features_df = pd.DataFrame(removed_features, columns=["Removed Features"])
removed_features_df.to_csv('XGBoost_removed_features.csv', index=False)

# XGBoost parameters
params = {
    'objective': 'multi:softprob',
    'num_class': len(class_list),
    'max_depth': 10,
    'eta': 0.1,
    'gamma': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'seed': 42
}

# Create dataset and train model
dtrain = xgb.DMatrix(X_train_selected, label=y_train_idx)
xgb_final_model = xgb.train(params, dtrain)

# Save the trained model in JSON format
xgb_final_model.save_model('XGBoost_selected_features_model.json')
